# Demographic Analysis
**Course:** GOVT 20.12 (Politics and AI), Dartmouth College

Parses the demographic analysis text files produced by `image_generation.ipynb`
and aggregates gender, race, and age counts per academic interest into a CSV
for use in `visualizations.R`.

**Inputs:** `academic_interest/<major>/analysis_<i>.txt`

**Outputs:** `folder_counts_detailed_academic.csv`

In [ ]:
# Imports and configuration

import os
import re
import pandas as pd
from collections import Counter, defaultdict
import string

ROOT_DIRECTORY = 'academic_interest'  # <-- Update this

# Regular expressions
age_pattern = re.compile(r'age:\s*(.+?)(?:,|\n|$)', re.IGNORECASE)
race_pattern = re.compile(r'race:\s*(.+?)(?:,|\n|$)', re.IGNORECASE)
gender_pattern = re.compile(r'gender:\s*(.+?)(?:,|\n|$)', re.IGNORECASE)

In [ ]:
# Text normalization helper functions for gender, race, and age fields

# Helper function to clean text (remove punctuation, lowercasing)
def clean_text(text):
    text = text.strip().lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

def normalize_gender(gender):
    gender = clean_text(gender)
    if 'female' in gender or 'woman' in gender or 'women' in gender:
        return 'Female'
    elif 'male' in gender:
        return 'Male'
    elif any(term in gender for term in ['androgynous', 'neutral', 'nonbinary', 'genderneutral']):
        return 'Non-binary/Androgynous'
    elif any(term in gender for term in ['indeterminate', 'not defined', 'unknown', 'cannot determine']):
        return 'UndeterminableGender'
    else:
        return 'OtherGender'

def normalize_race(race):
    race = clean_text(race)
    if 'caucasian' in race or 'white' in race:
        return 'White'
    elif 'asian' in race:
        return 'Asian'
    elif 'hispanic' in race or 'latino' in race:
        return 'Hispanic/Latino'
    elif 'middle eastern' in race:
        return 'Middle Eastern'
    elif 'black' in race or 'african' in race:
        return 'Black'
    elif any(term in race for term in ['unable to determine', 'indeterminate', 'unknown', 'not applicable', 'cannot determine']):
        return 'UndeterminableRace'
    else:
        return 'OtherRace'

def normalize_age(age):
    age = clean_text(age)
    if 'teen' in age:
        return 'Teen'
    elif '20' in age:
        return '20s'
    elif '30' in age:
        return '30s'
    elif '40' in age:
        return '40s'
    elif '50' in age:
        return '50s'
    elif '60' in age or '70' in age or 'elderly' in age or 'old' in age:
        return '60s+'
    elif 'middleaged' in age or 'middle aged' in age:
        return 'Middle-aged'
    elif 'young adult' in age or 'youthful' in age:
        return 'Young adult'
    elif any(term in age for term in ['indeterminate', 'not applicable', 'unknown', 'cannot determine']):
        return 'UndeterminableAge'
    else:
        return 'OtherAge'

In [57]:
# Parse analysis files and aggregate demographic counts per academic interest

# Dictionary to hold per-folder counters
folder_data = defaultdict(lambda: {'age': Counter(), 'race': Counter(), 'gender': Counter()})

# Walk through all folders and files
for subdir, _, files in os.walk(ROOT_DIRECTORY):
    subfolder_name = os.path.basename(subdir)
    if subfolder_name == os.path.basename(ROOT_DIRECTORY):
        continue  # skip root itself if it has files

    for filename in files:
        if filename.endswith('.txt'):
            filepath = os.path.join(subdir, filename)
            with open(filepath, 'r', encoding='utf-8') as file:
                content = file.read()

                # Extract fields
                age_match = age_pattern.search(content)
                race_match = race_pattern.search(content)
                gender_match = gender_pattern.search(content)

                if age_match:
                    normalized_age = normalize_age(age_match.group(1))
                    folder_data[subfolder_name]['age'][normalized_age] += 1
                if race_match:
                    normalized_race = normalize_race(race_match.group(1))
                    folder_data[subfolder_name]['race'][normalized_race] += 1
                if gender_match:
                    normalized_gender = normalize_gender(gender_match.group(1))
                    folder_data[subfolder_name]['gender'][normalized_gender] += 1

print("Parsed all analysis files.")

Saved improved detailed summary to folder_counts_detailed_academic.csv


In [ ]:

# Build summary DataFrame and save to CSV

# Explicit category lists ensure consistent column ordering in output CSV

# Find all unique category labels
all_ages = ['20s', '30s', '40s', '50s', 'Teen', '60s+', 'Middle-aged', 'Young adult', 'UndeterminableAge', 'OtherAge']
all_races = ['White', 'Black', 'Asian', 'Hispanic/Latino', 'Middle Eastern', 'UndeterminableRace', 'OtherRace']
all_genders = ['Male', 'Female', 'Non-binary/Androgynous', 'UndeterminableGender', 'OtherGender']

# Build rows
rows = []
for folder_name, counters in folder_data.items():
    row = {'name': folder_name}
    
    # Add age counts
    for age in all_ages:
        row[age] = counters['age'].get(age, 0)
    
    # Add race counts
    for race in all_races:
        row[race] = counters['race'].get(race, 0)
    
    # Add gender counts
    for gender in all_genders:
        row[gender] = counters['gender'].get(gender, 0)
    
    rows.append(row)

# Create DataFrame
df = pd.DataFrame(rows)

# Save to CSV
df.to_csv('folder_counts_detailed_academic.csv', index=False)

print("Saved improved detailed summary to folder_counts_detailed_academic.csv")